# 04 - D-Bus, KConfig, and plugin mapping


## Goal

Enumerate D-Bus interfaces + methods, cross-reference with KConfig groups + keys, and follow the `MaxResults` config key from declaration to the C++ class that reads it.


## Prerequisites

- Notebooks 01 and 02 complete — we re-extract the graph in the first code cell anyway.
- Mental model: D-Bus exposes behaviour; KConfig configures it.


## Environment bootstrap

This cell makes the notebook portable between a local checkout and Colab.

- **Local**: when the notebook lives inside the repo, we add the repo root to `sys.path`
  so the `src` package imports cleanly.
- **Colab**: the import will fail with `ModuleNotFoundError`. We catch that and print a
  one-line reminder showing the `git clone` the learner should run. We deliberately do
  **not** execute the clone for them — the lab policy is *recipes only, no auto-downloads*.


In [ ]:
import sys
from pathlib import Path

try:
    # Local checkout: walk up from the notebook to the repo root.
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / "src" / "common" / "paths.py").exists():
            if str(candidate) not in sys.path:
                sys.path.insert(0, str(candidate))
            break
    from src.common.paths import REPO_ROOT, MINI_REPO, ensure_dirs
    ensure_dirs()
    print(f"repo root: {REPO_ROOT}")
    print(f"mini repo: {MINI_REPO}")
except ModuleNotFoundError:
    print("`src` not importable. If you are on Colab, run this in a separate cell:")
    print("    !git clone https://example.invalid/kde_ontology_slm_lab.git")
    print("    %cd kde_ontology_slm_lab")
    print("Then re-run this cell. We will not auto-clone for you (lab policy:")
    print("recipes only, no auto-downloads).")


## 1. Re-extract the graph


In [ ]:
from src.common.paths import MINI_REPO
from src.repo_ingest.scanner import scan
from src.repo_ingest.cmake_reader import read_cmake
from src.repo_ingest.cpp_reader import read_cpp
from src.repo_ingest.qml_reader import read_qml
from src.repo_ingest.dbus_reader import read_dbus
from src.repo_ingest.kconfig_reader import read_kconfig
from src.repo_ingest.desktop_file_reader import read_desktop
from src.repo_ingest.log_reader import read_log
from src.ontology.extractor import (
    ExtractionBundle, from_cmake, from_cpp, from_qml, from_dbus,
    from_kconfig, from_desktop, from_log,
)
from src.ontology.schema import Entity
from src.common.ids import make_id
from src.graph.builder import build_graph

rep = scan(MINI_REPO)
b = ExtractionBundle()
rid = b.add_entity(Entity(id=make_id('Repository', MINI_REPO.name),
                          type='Repository', name=MINI_REPO.name,
                          source_path=str(MINI_REPO)))
for sf in rep.by_kind('cmake'): from_cmake(b, read_cmake(sf.path), rid)
for sf in rep.by_kind('cpp_header') + rep.by_kind('cpp_source'): from_cpp(b, read_cpp(sf.path))
for sf in rep.by_kind('qml'): from_qml(b, read_qml(sf.path))
for sf in rep.by_kind('dbus'): from_dbus(b, read_dbus(sf.path))
for sf in rep.by_kind('kconfig'): from_kconfig(b, read_kconfig(sf.path))
for sf in rep.by_kind('desktop'): from_desktop(b, read_desktop(sf.path))
for sf in rep.by_kind('log'): from_log(b, read_log(sf.path))
g = build_graph(b)


## 2. List D-Bus interfaces and their methods

Helper: `dbus_methods_of` (see `src/graph/queries.py`). Each `DbusInterface` node fans out via `EXPOSES_DBUS` edges to its methods and signals.


In [ ]:
from src.graph.queries import nodes_of_type, dbus_methods_of, neighbors_by_rel

for iface in nodes_of_type(g, 'DbusInterface'):
    iname = g.nodes[iface]['name']
    methods = dbus_methods_of(g, iface)
    all_exposed = neighbors_by_rel(g, iface, 'EXPOSES_DBUS')
    signals = [n for n in all_exposed if g.nodes[n].get('type') == 'DbusSignal']
    print(f'\nInterface: {iname}')
    print('  methods:', [g.nodes[m]['name'] for m in methods])
    print('  signals:', [g.nodes[s]['name'] for s in signals])


## 3. List KConfig groups and keys

A `ConfigFile` contains `ConfigGroup` nodes which contain `ConfigKey` nodes. The default value and type live on the key's properties (prefixed `prop_` in the NetworkX node attrs).


In [ ]:
for cf in nodes_of_type(g, 'ConfigFile'):
    print(f"\nConfig file: {g.nodes[cf]['name']}")
    for grp in neighbors_by_rel(g, cf, 'CONTAINS'):
        if g.nodes[grp].get('type') != 'ConfigGroup':
            continue
        print(f"  group: {g.nodes[grp]['name']}")
        for k in neighbors_by_rel(g, grp, 'CONTAINS'):
            d = g.nodes[k]
            print(f"    key  : {d['name']:18s} "
                  f"type={d.get('prop_type','?'):8s} "
                  f"default={d.get('prop_default','?')}")


## 4. Cross-reference: who reads `MaxResults`?

We invert the `READS_CONFIG` edge to find every `CppClass` that reads any `ConfigKey` named `MaxResults`. This is the same query the eval suite uses for the `eval:02:config-key` item.


In [ ]:
max_results_keys = [n for n, d in g.nodes(data=True)
                    if d.get('type') == 'ConfigKey' and d.get('name') == 'MaxResults']
print('MaxResults ConfigKey nodes:', max_results_keys)

for k in max_results_keys:
    readers = [u for u, _, kr in g.in_edges(k, keys=True) if kr == 'READS_CONFIG']
    for r in readers:
        d = g.nodes[r]
        print(f"  read by {d.get('type')} {d.get('name')} "
              f"at {d.get('source_path')}:{d.get('source_line')}")


## 5. The MaxResults end-to-end story

Putting it together:

1. `kconfig/minisearch.kcfg` declares `MaxResults` under the `[General]` group.
2. `KFileSearcher` reads `MaxResults` at runtime (extracted from the C++ source).
3. `KFileSearcher` is invoked from the QML `SearchView` (notebook 03).
4. The D-Bus `searchPath` method on `org.kde.minisearch` triggers a search,
   which respects the cap.

That is a complete causal chain from user-tunable config through to UI behaviour.


## Summary

You enumerated D-Bus surfaces and KConfig surfaces, and traced one config key from its declaration to its reader and onwards. Notebook 05 shifts gears to ask how a tokenizer sees all these KDE-specific identifiers.


## Exercises

1. List every `ConfigKey` that has **no** reader. Those are candidates for dead-config detection.
2. Add a `cancel` D-Bus method to the XML and re-run. Does it show up?
3. Write a query that returns the *intersection* of classes that read config and classes that log — those are the classes most worth tracing in production.
